# STEP 1

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import pdfplumber

def import_bank_transactions(file_path, file_type='csv'):
    """
    Import transactions from various file formats
    """
    if file_type == 'csv':
        df = pd.read_csv(file_path)
    elif file_type == 'excel':
        df = pd.read_excel(file_path)
    elif file_type == 'pdf':
        df = extract_from_pdf(file_path)
    
    # Standardize column names
    column_mapping = {
        'Date': 'date',
        'Transaction Date': 'date',
        'Description': 'description',
        'Memo': 'description',
        'Amount': 'amount',
        'Debit': 'debit',
        'Credit': 'credit'
    }
    
    df.rename(columns=column_mapping, inplace=True)
    
    # Convert date to datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Combine debit/credit into single amount column if needed
    if 'debit' in df.columns and 'credit' in df.columns:
        df['amount'] = df['credit'].fillna(0) - df['debit'].fillna(0)
        df.drop(['debit', 'credit'], axis=1, inplace=True)
    
    # Clean description field
    df['description'] = df['description'].str.strip().str.upper()
    
    return df.sort_values('date').reset_index(drop=True)

def extract_from_pdf(pdf_path):
    """
    Extract transaction data from PDF bank statements
    """
    transactions = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            # Parse text based on your bank's format
            # This is a simplified example
            lines = text.split('\n')
            for line in lines:
                # Custom parsing logic for your bank's format
                parts = line.split()
                if len(parts) >= 3:
                    transactions.append({
                        'date': parts[0],
                        'description': ' '.join(parts[1:-1]),
                        'amount': parts[-1]
                    })
    
    return pd.DataFrame(transactions)

# Import transactions from multiple sources
bank_transactions = import_bank_transactions("C://Harman\sample_bank_transactions.xlsx", 'excel')
credit_card = import_bank_transactions("C://Harman\sample_bank_transactions.xlsx", 'excel')

# Combine all transactions
all_transactions = pd.concat([bank_transactions, credit_card], ignore_index=True)
print(f"Total transactions imported: {len(all_transactions)}")

Total transactions imported: 20


In [2]:
all_transactions

,date,description,amount
0,2024-01-05,STAPLES OFFICE SUPPLIES STORE,-45.99
1,2024-01-06,UBER TRIP 8X4K2,-18.50
2,2024-01-07,STARBUCKS STORE #4521,-6.75
3,2024-01-08,PAYMENT RECEIVED INVOICE 1023,1500.00
4,2024-01-10,GOOGLE ADS CAMPAIGN,-250.00
5,2024-01-12,ELECTRIC COMPANY BILL,-120.34
6,2024-01-15,MONTHLY FEE,-12.00
7,2024-01-18,TRANSFER TO PERSONAL CHECKING,-800.00
8,2024-01-20,UNKNOWN VENDOR XYZ123,-99.99
9,2024-01-22,WIRE TRANSFER DEPOSIT,2200.00


# Loading files from same folder

In [4]:
import os
import glob
import pandas as pd


In [5]:
folder = r"C:\project\testingfiles" 

def get_file_type(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext == '.csv':
        return 'csv'
    elif ext in ('.xlsx', '.xls'):
        return 'excel'
    elif ext == '.pdf':
        return 'pdf'
    return None

def load_all_files_from_folder(folder_path):
    """
    Scan a folder for csv/excel/pdf files, load each with
    import_bank_transactions(), and merge them into one DataFrame.
    """
    patterns = ['*.csv', '*.xlsx', '*.xls', '*.pdf']
    all_files = []
    for pattern in patterns:
        all_files.extend(glob.glob(os.path.join(folder_path, pattern)))
        
    if not all_files:
        print(f"No CSV/Excel/PDF files found in {folder_path}")
        return pd.DataFrame()

    dfs = []
    for file_path in all_files:
        file_type = get_file_type(file_path)
        if file_type is None:
            continue
        try:
            df = import_bank_transactions(file_path, file_type)   # Function 1
            
            dfs.append(df)
            print(f"Loaded {len(df)} rows from {os.path.basename(file_path)}")
        except:
            print(f"Failed to load")

    if not dfs:
        return pd.DataFrame()

    merged = pd.concat(dfs, ignore_index=True)
    
    merged['amount'] = (
        merged['amount']
        .astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .str.strip()
    )
    merged['amount'] = pd.to_numeric(merged['amount'], errors='coerce')
    
    return merged.sort_values('date').reset_index(drop=True)


all_transactions = load_all_files_from_folder(folder)
print(f"\nTotal transactions imported: {len(all_transactions)}")



Loaded 10 rows from sample_bank_transactions.xlsx
Loaded 2 rows from test.pdf

Total transactions imported: 12


C:\Users\harma\AppData\Local\Temp\ipykernel_14368\506168436.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [6]:
type(all_transactions['amount'][1])

numpy.float64

In [7]:
all_transactions

,date,description,amount
0,2002-01-26,EXPENSE OFFICE SUPPLIES,1500.00
1,2003-01-26,INCOME SERVICES,1200.00
2,2024-01-05,STAPLES OFFICE SUPPLIES STORE,-45.99
3,2024-01-06,UBER TRIP 8X4K2,-18.50
4,2024-01-07,STARBUCKS STORE #4521,-6.75
5,2024-01-08,PAYMENT RECEIVED INVOICE 1023,1500.00
6,2024-01-10,GOOGLE ADS CAMPAIGN,-250.00
7,2024-01-12,ELECTRIC COMPANY BILL,-120.34
8,2024-01-15,MONTHLY FEE,-12.00
9,2024-01-18,TRANSFER TO PERSONAL CHECKING,-800.00


# Step 2

In [35]:
def create_categorization_rules():
    """
    Define rules for automatic transaction categorization
    """
    rules = {
        'Office Supplies': ['STAPLES', 'OFFICE DEPOT', 'AMAZON'],
        'Utilities': ['ELECTRIC', 'GAS COMPANY', 'WATER', 'INTERNET'],
        'Marketing': ['GOOGLE ADS', 'FACEBOOK', 'LINKEDIN', 'MAILCHIMP'],
        'Software': ['MICROSOFT', 'ADOBE', 'DROPBOX', 'SLACK'],
        'Meals & Entertainment': ['RESTAURANT', 'COFFEE', 'STARBUCKS', 'LUNCH'],
        'Travel': ['AIRLINE', 'HOTEL', 'UBER', 'LYFT', 'RENTAL CAR'],
        'Professional Services': ['LEGAL', 'ACCOUNTING', 'CONSULTING'],
        'Rent': ['LANDLORD', 'PROPERTY MANAGEMENT', 'RENT'],
        'Insurance': ['INSURANCE'],
        'Bank Fees': ['MONTHLY FEE', 'SERVICE CHARGE', 'ATM FEE'],
        'Client Payment': ['PAYMENT RECEIVED', 'INVOICE', 'DEPOSIT'],
        'Owner Draw': ['TRANSFER TO PERSONAL', 'OWNER WITHDRAWAL']
    }
    return rules

def categorize_transactions(df, rules):
    """
    Automatically categorize transactions based on description
    """
    df['category'] = 'Uncategorized'
    df['confidence'] = 0.0
    
    for category, keywords in rules.items():
        for keyword in keywords:
            mask = df['description'].str.contains(keyword, case=False, na=False)
            # Only categorize if not already categorized
            uncategorized = df['category'] == 'Uncategorized'
            df.loc[mask & uncategorized, 'category'] = category
            df.loc[mask & uncategorized, 'confidence'] = 0.9
    
    # Categorize by amount patterns
    df.loc[(df['amount'] > 0) & (df['category'] == 'Uncategorized'), 'category'] = 'Income'
    df.loc[(df['amount'] < 0) & (df['category'] == 'Uncategorized'), 'category'] = 'Other Expense'
    
    return df

def learn_from_manual_categories(df, manual_categories_file='C://Harman\manual_categories.csv'):
    """
    Learn from previously manually categorized transactions
    """
    try:
        manual = pd.read_csv(manual_categories_file)
        
        # Create a mapping of descriptions to categories
        category_map = {}
        for _, row in manual.iterrows():
            category_map[row['description']] = row['category']
        
        # Apply learned categories
        for desc, cat in category_map.items():
            mask = df['description'] == desc
            df.loc[mask, 'category'] = cat
            df.loc[mask, 'confidence'] = 1.0
    except FileNotFoundError:
        print("No manual categorization history found")
    
    return df

# Apply categorization
rules = create_categorization_rules()

all_transactions = categorize_transactions(all_transactions, rules) # 0.9
all_transactions = learn_from_manual_categories(all_transactions)   # 1.0

# Show categorization results
categorization_summary = all_transactions['category'].value_counts()
#print(categorization_summary)
print("\nCategorization Summary:")
print(categorization_summary)

# Flag low-confidence categorizations for review
needs_review = all_transactions[all_transactions['confidence'] < 0.9]
print(f"\nTransactions requiring manual review: {len(needs_review)}")


Categorization Summary:
category
Income                   2
Client Payment           2
Office Supplies          1
Travel                   1
Meals & Entertainment    1
Marketing                1
Utilities                1
Bank Fees                1
Owner Draw               1
Other Expense            1
Name: count, dtype: int64

Transactions requiring manual review: 3


# Step 3

In [36]:
def create_chart_of_accounts():
    """
    Define your chart of accounts structure
    """
    chart = {
        # Assets
        '1000': {'name': 'Cash - Checking', 'type': 'Asset'},
        '1010': {'name': 'Cash - Savings', 'type': 'Asset'},
        '1200': {'name': 'Accounts Receivable', 'type': 'Asset'},
        
        # Liabilities
        '2000': {'name': 'Accounts Payable', 'type': 'Liability'},
        '2100': {'name': 'Credit Card Payable', 'type': 'Liability'},
        
        # Equity
        '3000': {'name': 'Owner Equity', 'type': 'Equity'},
        '3100': {'name': 'Retained Earnings', 'type': 'Equity'},
        
        # Income
        '4000': {'name': 'Service Revenue', 'type': 'Income'},
        '4100': {'name': 'Product Sales', 'type': 'Income'},
        
        # Expenses
        '5000': {'name': 'Office Supplies', 'type': 'Expense'},
        '5100': {'name': 'Utilities', 'type': 'Expense'},
        '5200': {'name': 'Marketing', 'type': 'Expense'},
        '5300': {'name': 'Software', 'type': 'Expense'},
        '5400': {'name': 'Meals & Entertainment', 'type': 'Expense'},
        '5500': {'name': 'Travel', 'type': 'Expense'},
        '5600': {'name': 'Professional Services', 'type': 'Expense'},
        '5700': {'name': 'Rent', 'type': 'Expense'},
        '5800': {'name': 'Insurance', 'type': 'Expense'},
        '5900': {'name': 'Bank Fees', 'type': 'Expense'}
    }
    return chart

def map_category_to_account(category, chart_of_accounts):
    """
    Map transaction categories to general ledger accounts
    """
    category_mapping = {
        'Office Supplies': '5000',
        'Utilities': '5100',
        'Marketing': '5200',
        'Software': '5300',
        'Meals & Entertainment': '5400',
        'Travel': '5500',
        'Professional Services': '5600',
        'Rent': '5700',
        'Insurance': '5800',
        'Bank Fees': '5900',
        'Client Payment': '4000',
        'Income': '4000',
        'Other Expense': '5000'
    }
    
    return category_mapping.get(category, '5000')

def create_journal_entries(transactions_df, chart_of_accounts):
    """
    Create double-entry journal entries from transactions
    """
    journal_entries = []
    entry_number = 1
    
    for _, transaction in transactions_df.iterrows():
        account_code = map_category_to_account(transaction['category'], chart_of_accounts)
        amount = abs(transaction['amount'])
        
        if transaction['amount'] < 0:  # Expense
            # Debit expense account
            journal_entries.append({
                'entry_number': entry_number,
                'date': transaction['date'],
                'account_code': account_code,
                'account_name': chart_of_accounts[account_code]['name'],
                'debit': amount,
                'credit': 0,
                'description': transaction['description']
            })
            # Credit cash account
            journal_entries.append({
                'entry_number': entry_number,
                'date': transaction['date'],
                'account_code': '1000',
                'account_name': 'Cash - Checking',
                'debit': 0,
                'credit': amount,
                'description': transaction['description']
            })
        else:  # Income
            # Debit cash account
            journal_entries.append({
                'entry_number': entry_number,
                'date': transaction['date'],
                'account_code': '1000',
                'account_name': 'Cash - Checking',
                'debit': amount,
                'credit': 0,
                'description': transaction['description']
            })
            # Credit revenue account
            journal_entries.append({
                'entry_number': entry_number,
                'date': transaction['date'],
                'account_code': account_code,
                'account_name': chart_of_accounts[account_code]['name'],
                'debit': 0,
                'credit': amount,
                'description': transaction['description']
            })
        
        entry_number += 1
    
    return pd.DataFrame(journal_entries)

# Create journal entries
chart = create_chart_of_accounts()
journal = create_journal_entries(all_transactions, chart)
print(f"\nJournal entries created: {len(journal)}")


Journal entries created: 24


In [37]:
journal

,entry_number,date,account_code,account_name,debit,credit,description
0,1,2002-01-26,1000,Cash - Checking,1500.00,0.00,EXPENSE OFFICE SUPPLIES
1,1,2002-01-26,4000,Service Revenue,0.00,1500.00,EXPENSE OFFICE SUPPLIES
2,2,2003-01-26,1000,Cash - Checking,1200.00,0.00,INCOME SERVICES
3,2,2003-01-26,4000,Service Revenue,0.00,1200.00,INCOME SERVICES
4,3,2024-01-05,5000,Office Supplies,45.99,0.00,STAPLES OFFICE SUPPLIES STORE
5,3,2024-01-05,1000,Cash - Checking,0.00,45.99,STAPLES OFFICE SUPPLIES STORE
6,4,2024-01-06,5500,Travel,18.50,0.00,UBER TRIP 8X4K2
7,4,2024-01-06,1000,Cash - Checking,0.00,18.50,UBER TRIP 8X4K2
8,5,2024-01-07,5400,Meals & Entertainment,6.75,0.00,STARBUCKS STORE #4521
9,5,2024-01-07,1000,Cash - Checking,0.00,6.75,STARBUCKS STORE #4521


# Step 4: Generate Financial Statements

In [38]:
def generate_income_statement(journal_df, start_date, end_date):
    """
    Create an automated income statement
    """
    # Filter by date range
    period_journal = journal_df[
        (journal_df['date'] >= start_date) & 
        (journal_df['date'] <= end_date)
    ]

    
    # Calculate income
    income_accounts = period_journal[
        period_journal['account_code'].str.startswith('4')
    ]
    total_income = income_accounts['credit'].sum() - income_accounts['debit'].sum()

    
    # Calculate expenses by category
    expense_accounts = period_journal[
        period_journal['account_code'].str.startswith('5')
    ]
    expenses_by_account = expense_accounts.groupby('account_name').agg({
        'debit': 'sum',
        'credit': 'sum'
    })

 
    expenses_by_account['net'] = expenses_by_account['debit'] - expenses_by_account['credit']

    total_expenses = expenses_by_account['net'].sum()


    # Calculate net income
    net_income = total_income - total_expenses
    
    # Format income statement
    income_statement = pd.DataFrame({
        'Account': ['REVENUE', '  Total Revenue', '', 'EXPENSES'] + 
                   ['  ' + idx for idx in expenses_by_account.index] +
                   ['  Total Expenses', '', 'NET INCOME'],
        'Amount': ['', f"${total_income:,.2f}", ''] + 
                  [''] + 
                  [f"${amt:,.2f}" for amt in expenses_by_account['net']] +
                  [f"${total_expenses:,.2f}", '', f"${net_income:,.2f}"]
    })

    return income_statement, net_income



In [39]:
journal

,entry_number,date,account_code,account_name,debit,credit,description
0,1,2002-01-26,1000,Cash - Checking,1500.00,0.00,EXPENSE OFFICE SUPPLIES
1,1,2002-01-26,4000,Service Revenue,0.00,1500.00,EXPENSE OFFICE SUPPLIES
2,2,2003-01-26,1000,Cash - Checking,1200.00,0.00,INCOME SERVICES
3,2,2003-01-26,4000,Service Revenue,0.00,1200.00,INCOME SERVICES
4,3,2024-01-05,5000,Office Supplies,45.99,0.00,STAPLES OFFICE SUPPLIES STORE
5,3,2024-01-05,1000,Cash - Checking,0.00,45.99,STAPLES OFFICE SUPPLIES STORE
6,4,2024-01-06,5500,Travel,18.50,0.00,UBER TRIP 8X4K2
7,4,2024-01-06,1000,Cash - Checking,0.00,18.50,UBER TRIP 8X4K2
8,5,2024-01-07,5400,Meals & Entertainment,6.75,0.00,STARBUCKS STORE #4521
9,5,2024-01-07,1000,Cash - Checking,0.00,6.75,STARBUCKS STORE #4521


In [40]:
def generate_balance_sheet(journal_df, as_of_date):
    """
    Create an automated balance sheet
    """
    # Filter up to date
    period_journal = journal_df[journal_df['date'] <= as_of_date]
    
    # Calculate balances by account type
    balances = period_journal.groupby(['account_code', 'account_name']).agg({
        'debit': 'sum',
        'credit': 'sum'
    })
    balances['balance'] = balances['debit'] - balances['credit']
    
    # Separate by account type
    assets = balances[balances.index.get_level_values(0).str.startswith('1')]
    liabilities = balances[balances.index.get_level_values(0).str.startswith('2')]
    equity = balances[balances.index.get_level_values(0).str.startswith('3')]
    
    
    total_assets = assets['balance'].sum()
    total_liabilities = abs(liabilities['balance'].sum())
    total_equity = abs(equity['balance'].sum())
    
    return {
        'Total Assets': total_assets,
        'Total Liabilities': total_liabilities,
        'Total Equity': total_equity,
        'Assets': assets,
        'Liabilities': liabilities,
        'Equity': equity
    }


In [41]:
start_date = pd.to_datetime('2024-01-07')
end_date = pd.to_datetime('2024-01-18')

income_stmt, net_income = generate_income_statement(journal, start_date, end_date)
balance_sheet = generate_balance_sheet(journal, end_date)

print("\n" + "="*50)
print("INCOME STATEMENT")
print(f"{start_date.strftime('%B %Y')}")
print("="*50)
print(income_stmt.to_string(index=False))

print("\n" + "="*50)
print("BALANCE SHEET")
print(f"As of {end_date.strftime('%B %d, %Y')}")
print("="*50)
print(f"Total Assets: ${balance_sheet['Total Assets']:,.2f}")
print(f"Total Liabilities: ${balance_sheet['Total Liabilities']:,.2f}")
print(f"Total Equity: ${balance_sheet['Total Equity']:,.2f}")


INCOME STATEMENT
January 2024
                Account    Amount
                REVENUE          
          Total Revenue $1,500.00
                                 
               EXPENSES          
              Bank Fees    $12.00
              Marketing   $250.00
  Meals & Entertainment     $6.75
        Office Supplies   $800.00
              Utilities   $120.34
         Total Expenses $1,189.09
                                 
             NET INCOME   $310.91

BALANCE SHEET
As of January 18, 2024
Total Assets: $2,946.42
Total Liabilities: $0.00
Total Equity: $0.00
